# Monte Carlo Algorithm and Bias

## Introduction

Monte Carlo methods solve problems through random sampling. In NSGABLACK, combining Monte Carlo with the Bias System can significantly improve search efficiency.

## Basic Monte Carlo Method

### Random Sampling Strategy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from solvers.monte_carlo import MonteCarloSolver
from bias.bias_v2 import BiasSystemV2

# Basic Monte Carlo example
def monte_carlo_optimization(func, bounds, n_samples=1000):
    """Basic Monte Carlo optimization"""
    best_x = None
    best_value = float('inf')
    
    for _ in range(n_samples):
        # Random sampling
        x = np.random.uniform(bounds[0], bounds[1])
        value = func(x)
        
        if value < best_value:
            best_value = value
            best_x = x
    
    return best_x, best_value

# Test function: Sphere function
def sphere_function(x):
    if isinstance(x, np.ndarray):
        return np.sum(x**2)
    return x**2

# Basic Monte Carlo optimization
best_x, best_val = monte_carlo_optimization(
    lambda x: sphere_function(np.array([x])), 
    bounds=[-10, 10],
    n_samples=10000
)

print(f"Best solution: {best_x:.4f}")
print(f"Best value: {best_val:.4f}")

## Biased Monte Carlo Method

Combine Monte Carlo with bias system to improve sampling efficiency:

In [ ]:
class BiasedMonteCarlo:
    """Monte Carlo method with bias"""
    
    def __init__(self, bias_system=None):
        self.bias_system = bias_system
        self.history = []
        self.best_solutions = []
        
    def optimize(self, func, bounds, dim=1, n_samples=10000):
        """Biased Monte Carlo optimization"""
        best_x = None
        best_value = float('inf')
        
        # Initial sampling
        for i in range(n_samples):
            # Generate sample
            if i < n_samples // 2:
                # First half: uniform sampling
                x = np.random.uniform(bounds[0], bounds[1], dim)
            else:
                # Second half: biased sampling
                x = self._biased_sample(bounds, dim)
            
            value = func(x)
            
            if value < best_value:
                best_value = value
                best_x = x.copy()
                self.best_solutions.append(x.copy())
            
            self.history.append(value)
        
        return best_x, best_value
    
    def _biased_sample(self, bounds, dim):
        """Generate biased sample"""
        if len(self.best_solutions) > 0:
            # Sample around best solutions
            base = self.best_solutions[np.random.randint(len(self.best_solutions))]
            noise = np.random.normal(0, 0.1, dim)
            x = base + noise
            # Ensure within bounds
            x = np.clip(x, bounds[0], bounds[1])
            return x
        else:
            return np.random.uniform(bounds[0], bounds[1], dim)

# Compare unbiased and biased Monte Carlo
bounds = [-5, 5]

# Unbiased Monte Carlo
best_x_unbiased, best_val_unbiased = monte_carlo_optimization(
    lambda x: sphere_function(np.array([x])),
    bounds,
    n_samples=5000
)

# Biased Monte Carlo
bmc = BiasedMonteCarlo()
best_x_biased, best_val_biased = bmc.optimize(
    lambda x: sphere_function(np.array([x])),
    bounds,
    n_samples=5000
)

print(f"Unbiased MC - Best: {best_val_unbiased:.4f}")
print(f"Biased MC - Best: {best_val_biased:.4f}")

## Monte Carlo with NSGABLACK Bias System

Use the built-in bias system:

In [ ]:
# Create Monte Carlo solver with bias system
bias_system = BiasSystemV2()
mc_solver = MonteCarloSolver(
    bias_system=bias_system,
    sampling_strategy="adaptive"
)

# Optimize 2D Rastrigin function
def rastrigin_function(x):
    A = 10
    n = len(x)
    return A * n + sum([(xi**2 - A * np.cos(2 * np.pi * xi)) for xi in x])

# Run optimization
result = mc_solver.solve(
    objective_func=rastrigin_function,
    bounds=[(-5.12, 5.12), (-5.12, 5.12)],
    n_samples=10000,
    bias_strength=0.3
)

print(f"Optimal solution: {result.best_solution}")
print(f"Optimal value: {result.best_value}")

## Advanced Applications

### 1. Multi-Start Monte Carlo with Bias

In [ ]:
class MultiStartMonteCarlo:
    """Multi-start Monte Carlo with bias"""
    
    def __init__(self, n_starts=5, bias_system=None):
        self.n_starts = n_starts
        self.bias_system = bias_system
        self.results = []
    
    def optimize(self, func, bounds, dim, samples_per_start=1000):
        """Multi-start optimization"""
        global_best = None
        global_best_val = float('inf')
        
        for start in range(self.n_starts):
            # Random start point
            start_point = np.random.uniform(bounds[0], bounds[1], dim)
            
            # Monte Carlo from this start point
            best_x, best_val = self._local_monte_carlo(
                func, start_point, bounds, samples_per_start
            )
            
            if best_val < global_best_val:
                global_best_val = best_val
                global_best = best_x
            
            self.results.append((best_x, best_val))
            
            print(f"Start {start + 1}: Best value = {best_val:.4f}")
        
        return global_best, global_best_val
    
    def _local_monte_carlo(self, func, start_point, bounds, n_samples):
        """Local Monte Carlo search around start point"""
        best_x = start_point.copy()
        best_val = func(start_point)
        radius = (bounds[1] - bounds[0]) * 0.1  # Local search radius
        
        for _ in range(n_samples):
            # Sample around start point
            x = start_point + np.random.normal(0, radius, len(start_point))
            x = np.clip(x, bounds[0], bounds[1])
            
            val = func(x)
            
            if val < best_val:
                best_val = val
                best_x = x.copy()
        
        return best_x, best_val

# Test multi-start Monte Carlo
msmc = MultiStartMonteCarlo(n_starts=5)
best_x, best_val = msmc.optimize(
    rastrigin_function,
    bounds=[-5, 5],
    dim=2,
    samples_per_start=2000
)

print(f"\nGlobal best: {best_val:.4f}")
print(f"Global best solution: {best_x}")

### 2. Monte Carlo with Local Search

In [ ]:
def hill_climbing(func, x, step_size=0.1, max_iter=100):
    """Simple hill climbing local search"""
    current_x = x.copy()
    current_val = func(x)
    
    for _ in range(max_iter):
        improved = False
        
        # Try all directions
        for i in range(len(current_x)):
            for direction in [-1, 1]:
                new_x = current_x.copy()
                new_x[i] += direction * step_size
                new_val = func(new_x)
                
                if new_val < current_val:
                    current_x = new_x
                    current_val = new_val
                    improved = True
        
        if not improved:
            break
    
    return current_x, current_val

class MonteCarloWithLocalSearch:
    """Monte Carlo combined with local search"""
    
    def optimize(self, func, bounds, dim, n_samples=1000):
        best_x = None
        best_val = float('inf')
        
        for _ in range(n_samples):
            # Random sample
            x = np.random.uniform(bounds[0], bounds[1], dim)
            
            # Local search from this point
            refined_x, refined_val = hill_climbing(func, x)
            
            if refined_val < best_val:
                best_val = refined_val
                best_x = refined_x
        
        return best_x, best_val

# Test Monte Carlo with local search
mc_ls = MonteCarloWithLocalSearch()
best_x, best_val = mc_ls.optimize(
    rastrigin_function,
    bounds=[-5, 5],
    dim=2,
    n_samples=100
)

print(f"Monte Carlo + Local Search Best: {best_val:.4f}")
print(f"Best solution: {best_x}")

## Performance Comparison

Compare different Monte Carlo variants:

In [ ]:
# Performance comparison
methods = {
    "Basic MC": lambda: monte_carlo_optimization(rastrigin_function, [-5, 5], 1000),
    "Biased MC": lambda: BiasedMonteCarlo().optimize(rastrigin_function, [-5, 5], 1, 1000),
    "Multi-start MC": lambda: MultiStartMonteCarlo(3).optimize(rastrigin_function, [-5, 5], 2, 333),
    "MC + Local Search": lambda: MonteCarloWithLocalSearch().optimize(rastrigin_function, [-5, 5], 2, 100)
}

results = {}
for method_name, method_func in methods.items():
    best_x, best_val = method_func()
    results[method_name] = best_val
    print(f"{method_name}: {best_val:.4f}")

# Plot comparison
plt.figure(figsize=(10, 6))
plt.bar(results.keys(), results.values())
plt.ylabel('Best Value')
plt.title('Monte Carlo Methods Comparison')
plt.xticks(rotation=45)
plt.show()

## Best Practices

1. **Choose appropriate sampling strategy**:
   - Uniform sampling for simple problems
   - Biased sampling for complex landscapes

2. **Use bias wisely**:
   - Don't over-bias early in search
   - Increase bias gradually

3. **Combine with local search**:
   - Improves final solution quality
   - Reduces need for large sample sizes

## Summary

Monte Carlo methods in NSGABLACK:

- Simple and robust
- Can be enhanced with bias system
- Suitable for global exploration
- Good starting point for local optimization

## Next Tutorial

Next: [NSGA-II Multi-objective Optimization](04_NSGA2_Algorithm_and_Bias_Application.ipynb)